# Fine-tune Qwen 3.5 4B for CTF/Coding Tasks

Uses Unsloth with 4-bit QLoRA on free Google Colab T4 GPU.

**Setup:**
- Runtime → Change runtime type → **T4 GPU**
- Drag-and-drop `train.jsonl` into Colab (or upload via cell 2)
- Run cells top-to-bottom

In [ ]:
# Cell 1: Install Unsloth
%%capture
import os, re, subprocess
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch
    v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
    !pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
import torch
torch._dynamo.config.recompile_limit = 64
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Upload training data
# Option A: Drag-and-drop train.jsonl into Colab files panel before running
# Option B: Use this cell to upload from local
from google.colab import files
uploaded = files.upload()  # Upload train.jsonl
print(f"Uploaded files: {list(uploaded.keys())}")

In [ ]:
# Cell 3: Load and verify dataset
from datasets import load_dataset
import json

# Load the JSONL file
dataset = load_dataset("json", data_files={"train": "/content/train.jsonl"}, split="train")
print(f"Dataset size: {len(dataset)} examples")
print(f"\nFirst example:")
print(f"  System: {dataset[0]['messages'][0]['content'][:100]}...")
print(f"  User: {dataset[0]['messages'][1]['content'][:100]}...")
print(f"  Assistant: {dataset[0]['messages'][2]['content'][:100]}...")

In [ ]:
# Cell 4: Load model with 4-bit QLoRA
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/Qwen3.5-4B"
MAX_SEQ_LENGTH = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,  # 4-bit QLoRA for T4
    dtype=None,
)
print(f"Model loaded: {MODEL_NAME}")
print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
# Cell 5: Apply chat template
from unsloth import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="chatml")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"Dataset formatted: {len(dataset)} examples")

In [ ]:
# Cell 6: Configure LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% less VRAM
    random_state=3407,
    max_seq_length=MAX_SEQ_LENGTH,
)
print(f"LoRA configured: r=16, alpha=16")

In [ ]:
# Cell 7: Train
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,  # effective batch = 4
        warmup_steps=10,
        num_train_epochs=3,  # Adjust as needed
        learning_rate=2e-4,
        logging_steps=10,
        output_dir="/content/outputs",
        optim="adamw_8bit",
        seed=3407,
        save_strategy="epoch",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        dataset_text_field="text",
        report_to="none",
    ),
)

print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
# Cell 8: Save model
OUTPUT_DIR = "/content/outputs/qwen4b-ctf"

# Save LoRA adapter
model.save_pretrained(f"{OUTPUT_DIR}/lora")
tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora")
print(f"✓ LoRA saved to {OUTPUT_DIR}/lora")

# Export to GGUF (for llama.cpp/Ollama)
model.save_pretrained_gguf(f"{OUTPUT_DIR}/gguf", tokenizer, quantization_method="q4_k_m")
print(f"✓ GGUF saved to {OUTPUT_DIR}/gguf")

# Export merged model (16-bit SafeTensors)
model.save_pretrained_merged(f"{OUTPUT_DIR}/merged", tokenizer, save_method="merged_16bit")
print(f"✓ Merged model saved to {OUTPUT_DIR}/merged")

In [ ]:
# Cell 9: Download results
from google.colab import files
import shutil

OUTPUT_DIR = "/content/outputs/qwen4b-ctf"

# Zip everything for easy download
shutil.make_archive("/content/qwen4b-ctf-output", "zip", "/content/outputs")
files.download("/content/qwen4b-ctf-output.zip")
print("Download started!")

In [ ]:
# Cell 10: (Optional) Test the model
from transformers import TextStreamer

# Test prompt
messages = [
    {"role": "system", "content": "You are an expert CTF player."},
    {"role": "user", "content": "Explain how a buffer overflow exploit works."},
    ]

input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

text_streamer = TextStreamer(tokenizer, skip_prompt=True)
output = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=512,
    temperature=1.0,
    top_p=0.95,
    top_k=64,
)